# Stellar Neural Network Interpolation - ShinkaEvolve Launcher

**Task**: Optimize a FLAX neural network for stellar evolution track interpolation
**Objective**: Max fractional residual for Delta Nu < 0.00079 (Kepler-LEGACY)

Missing imports:
- jax
- flax
- tables
- pandas (unsure)

In [1]:
import warnings
import dotenv
import os

# Suppress third-party warnings before importing shinka
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

# Check if ShinkaEvolve is importable
try:
    import shinka
except ImportError:
    print("shinka not found, make sure to open this notebook with the ShinkaEvolve environment active")

# Find the .env file
env_path = dotenv.find_dotenv()

# Make sure that there's a .env file
assert env_path, ".env not found, please add it to the root of this project."

# Find the repository root assuming that's where the .env file is
repo_path = os.path.dirname(env_path)
activate_path = os.path.join(repo_path, '.venv/bin/activate')

# Print out where the .env file and repo root are
print("> loaded .env from {}".format(env_path))
print("> repo root located at {}".format(repo_path))

# Load the environment variables
dotenv.load_dotenv()

# Check that OPENROUTER_API_KEY is set in the .env file
if os.environ.get("OPENROUTER_API_KEY"):
    print("> OPENROUTER_API_KEY found")
else:
    print("> WARNING: OPENROUTER_API_KEY not set — add it to .env file")

> loaded .env from /Users/mp2557/Documents/Research/Tutorial_Shinka/.env
> repo root located at /Users/mp2557/Documents/Research/Tutorial_Shinka
> OPENROUTER_API_KEY found


In [2]:
import evaluate
import initial_program

# Test the initial program for circle packing
data_bundles = evaluate.load_and_prep_data()
output = initial_program.run_training(**data_bundles)

# Check if the the program outputs a valid circle packing
valid, msg = evaluate.validate_stellar(output)

# Assert check
assert valid, f"> Initial test FAILED: {msg}"

print(f"> Initial test: PASSED  (loss={output[2]:.6f})")

KeyError: 2

In [25]:
import datetime as dt

# A description of the task ShinkaEvolve is solving to be sent as an LLM prompt.
TASK_SYS_MSG = """
Your task is to perform interpolation using a jax-based neural network (FLAX) over a grid of stellar evolution tracks generated by ASTEC. 
The absolute-valued fractional residuals for the test set must be less than 0.00079 for 'delta_nu'.

Key directions to explore:
1. Various hyperparameter values (learning rate, weight decay, dropout, learning rate scheduler, etc.)
2. Various architectures, incuding changing the batch size, hidden dimensions, neural network layer types, etc.
3. You can try k-fold validation to improve

Constraints:
1. Make sure that you use a jax-based neural network (FLAX) and not another type of machine-learning interpolator.

Be creative!
"""

# Number of generations to run in this experiment
NUM_GENERATIONS = 20

# Results will be stored in a directory "circpack_<CURRENT DATE-TIME>"
experiment_name = "stellar_" + dt.datetime.now().strftime("%Y%m%d_%H%M%S")

# Set RESULTS_DIR
RESULTS_DIR = "results/{}".format(experiment_name)

# Print out the path
print(f"> Results dir: {RESULTS_DIR}")

> Results dir: results/stellar_20260421_102648


In [26]:
# Set cost if you want to try this out!
cost = 0

# Define the MAX_API_COST variable
MAX_API_COST = cost or None

# Has my cost been set?
print('> Cost limit? {}'.format(MAX_API_COST))

# Define all LLM related hyperparameters.
LLM_MODELS = [
    "openrouter/anthropic/claude-haiku-4-5",
    "openrouter/openai/gpt-5.1-codex-mini",
    "openrouter/openai/gpt-5-nano",
    "openrouter/google/gemini-3-flash-preview"
]

META_LLM_MODELS = ["openrouter/openai/o4-mini"]

NOVELTY_LLM_MODELS = ["openrouter/openai/o4-mini"]

EMBEDDING_MODEL = "openrouter/openai/text-embedding-3-small"

> Cost limit? None


In [27]:
# Import from config objects from the shinka library
from shinka.core import EvolutionConfig
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig


# Set the evolution config object
evo_config = EvolutionConfig(init_program_path="initial_program.py",
                             task_sys_msg=TASK_SYS_MSG,
                             num_generations=NUM_GENERATIONS,
                             results_dir=RESULTS_DIR,
                             max_api_costs=MAX_API_COST,
                             llm_models=LLM_MODELS,
                             meta_llm_models=META_LLM_MODELS,
                             novelty_llm_models=NOVELTY_LLM_MODELS,
                             embedding_model=EMBEDDING_MODEL)


# Set the job config. ACTIVATE_SCRIPT is a parameter which tells what virtual
# environment ShinkaEvolve will run evolved programs in.
job_config = LocalJobConfig(eval_program_path="evaluate.py",
                            activate_script=activate_path)

# If you're using Conda to manage your virtual environment, you will need to
# instead set CONDA_ENV. Uncomment this line to do that
# job_config = LocalJobConfig(eval_program_path=EVAL_PROGRAM_PATH,
#                             conda_env="shinka_ai4sd26")

# Number of islands is a hyperparameter which affects the evolution algorithm
# run by ShinkaEvolve. This also affects the visualization. See the guide
# at `recipes/hyperparameters.md` for more information.
db_config = DatabaseConfig(num_islands=2)

In [28]:
from shinka.core import ShinkaEvolveRunner

from time import perf_counter

# Create the ShinkaEvolveRunner object.
runner = ShinkaEvolveRunner(
    evo_config=evo_config,
    job_config=job_config,
    db_config=db_config,
    verbose=True,
)

# Store the starting wallclock time
tic = perf_counter()

# Run ShinkaEvolve by calling `run_async`
await runner.run_async()

# Store the stop wallclock time
toc = perf_counter()

# Print out information
print(f"> Evolution completed in {toc - tic:.1f} s")
print(f"> Results saved to: {runner.results_dir}")

  @@@@@@@@@@@@@@@@@@@@@      ░██████╗██╗░░██╗██╗███╗░░██╗██╗░░██╗░█████╗░
  @                   @      ██╔════╝██║░░██║██║████╗░██║██║░██╔╝██╔══██╗
  @          @        @      ╚█████╗░███████║██║██╔██╗██║█████═╝░███████║
  @    @@   @@  @@    @      ░╚═══██╗██╔══██║██║██║╚████║██╔═██╗░██╔══██║
  @   @     @    @@   @      ██████╔╝██║░░██║██║██║░╚███║██║░╚██╗██║░░██║
  @    @@  @    @     @      ╚═════╝░╚═╝░░╚═╝╚═╝╚═╝░░╚══╝╚═╝░░╚═╝╚═╝░░╚═╝
  @        @          @      @@@@@@@@@@@@@@@
  @                   @   @@                 @@@@@
  @@@@@@@@@@@@@@@@@@@@ @@                       @  @@                 █▀▀
                      @                          @@  @                ██▄
                    @      @@                      @  @@
                   @       @         @              @   @             █░█
                   @                 @               @  @             ▀▄▀
                     @@@@@          @     @           @  @
                      @            @          @ 

2026-04-21 10:26:53 - shinka.core.async_runner - INFO - 🖥️  System resources detected:

2026-04-21 10:26:53 - shinka.core.async_runner - INFO -    • CPU cores: 8

2026-04-21 10:26:53 - shinka.core.async_runner - INFO -    • Memory: 16.0 GB

2026-04-21 10:26:53 - shinka.core.async_runner - INFO - 🔧 Concurrency settings:

...

2026-04-21 10:28:21 - shinka.core.async_runner - INFO - Proposal target=2 (sampling_ewma=43.50s,                   
evaluation_ewma=3.84s, timing_samples=2, active_proposals=1, running_jobs=0)

2026-04-21 10:28:35 - shinka.core.async_runner - INFO - 🔍 Meta task check: completed_gens=3, target=20,           
pending_work=17, running_eval_jobs=0, running_proposal_jobs=1, api_costs=$0.0322, should_stop=False,               
is_stuck=False, stuck_count=0, time_since_progress=14.3s

2026-04-21 10:29:15 - shinka.database.async_dbase - INFO - 🔧 Closing async database with monitoring...

2026-04-21 10:29:15 - shinka.database.async_dbase - INFO - Async database closed

CancelledError: 